<a href="https://colab.research.google.com/github/babessell1/GWC_Test/blob/main/2D_Animations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ipympl
from google.colab import output
output.enable_custom_widget_manager()
# restart the runtime after running this block!

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
%matplotlib ipympl

In [ ]:
# variable cheat-sheet
# ==========================
# y(x,t)  - instantaneous wave value (what we visualize)
# A       - amplitude (peak value)
# x, z    - spatial coordinates (1D or 2D)
# Δx      - displacement / source position shift
# λ       - wavelength (distance per cycle)
# k       - wave number = 2π / λ (radians per unit length)
# f       - frequency in cycles/second (Hz)
# ω       - angular frequency = 2π f (radians/second)
# v       - wave speed (length / second). For simple non-dispersive waves: ω = k * v
# φ       - phase offset (radians). If provided in degrees, convert with np.deg2rad()
# r       - distance from a point source to the field point (sqrt(dx^2 + dy^2) in 2D)
# t       - time (seconds)
# i       - the imaginary unit, i² = −1. (it lets us represent a 2D arrow.)

sometimes, we, as programmers, need to use "tricks" to make it easier for the computer to do what we want.
this usually happens when we need the computer to do something many times. in our case, we want to animate our sine wave interactions, but taking sines and square roots is actually quite slow. so we are first going to rewirte our sine wave equation in a way that is faster for the computer to solve.


#Phasor trick
Imagine every sine wave as a little spinning arrow (like a clock hand). The wave value is just the arrow’s vertical shadow. Instead of recomputing lots of sines every frame, we pre‑draw the arrows and spin them — cheap and fast.

The clock‑hand picture (no heavy math)

Take an arrow of length A. Put its tail at the origin.  
Point it at angle θ. The arrow’s vertical shadow (up/down) = A · sin(θ).  
Now make that arrow rotate smoothly — its vertical shadow wiggles in time → it makes a sine wave.

So: sine = vertical shadow of a spinning arrow.

In [ ]:
# google "phasor trick, or Euler's formula"

# Euler’s formula:
# e^{iθ} = cos θ + i·sin θ  ⇒  sin θ = Im(e^{iθ}) ... (Im means the imaginary part!)

# Start with the usual wave:
# y(x,t) = A · sin(kx − ωt + φ)

# Rewrite it using eulers formula:
# y(x,t) = Im{ A · e^{i(kx − ωt + φ)} }

# Factor time out:
# y(x,t) = Im{ [A · e^{i(kx + φ)}] · e^{−iωt} }

# Define the spatial arrow (phasor):
# C(x) = A · e^{i(kx + φ)}
#  — this is a fixed arrow at each x.

# Now y(x,t) = Im{ C(x) · e^{−iωt} }
#  — rotate every arrow by the same amount e^{−iωt} and take its vertical shadow

In [ ]:
assert 3 == 3, "This is not true"

In [ ]:
# ==============================================================================
# Part A — Small 1-D sanity check: traditional sin() vs phasor formulation
# ==============================================================================

def sine_wave_traditional(x, amplitude=1.0, phase_deg=0.0, displacement=0.0, wavelength=1.0):
    """
    The usual real-valued sine definition for 1-D demonstrations:
      y(x) = A * sin( k*(x - displacement) + phi )
    """
    phi = np.deg2rad(float(phase_deg))
    k = 2.0 * np.pi / float(wavelength)
    return amplitude * np.sin(k * (x - displacement) + phi)


def sine_wave_phasor(x, amplitude=1.0, phase_deg=0.0, displacement=0.0, wavelength=1.0, time=0.0, wave_speed=1.0):
    """
    The phasor-based version: build the complex phasor C(x) = A * exp(i*(k*(x-dx) + phi))
    and then return the instantaneous real wave y(x,t) = Im( C(x) * exp(-i * omega * t) ).
    This shows the algebraic equivalence to the traditional sine.
    """
    phi = np.deg2rad(float(phase_deg))
    k = 2.0 * np.pi / float(wavelength)
    omega = 2.0 * np.pi * wave_speed / float(wavelength)
    C = amplitude * np.exp(1j * (k * (x - displacement) + phi))   # complex spatial phasor
    y = np.imag(C * np.exp(-1j * omega * time))                   # rotate in time, then take imaginary part
    return y

# double check!
# are the functions equivalent for t=0 (and at any t, if we include time)?
xs = np.linspace(0, 1, 50)
y_trad = sine_wave_traditional(xs, amplitude=1.23, phase_deg=30, displacement=0.1, wavelength=0.3)
y_phas = sine_wave_phasor(xs, amplitude=1.23, phase_deg=30, displacement=0.1, wavelength=0.3, time=0.0)
assert np.allclose(y_trad, y_phas, atol=1e-12), "Traditional and phasor versions must match (t=0)."

In [ ]:
import timeit

# timeit.timeit runs the function 'number' times (default is often 1,000,000)
# and returns the total time taken as a float.
total_time = timeit.timeit(stmt="sine_wave_traditional(1, 1, 2, 10)", setup="from __main__ import sine_wave_traditional", number=1_000_000)
average_time = total_time / 1_000_000

print(f"Average time over 1,000,000 runs: {average_time:.6f} seconds")


Average time over 1,000,000 runs: 0.000002 seconds


In [ ]:
import timeit

# timeit.timeit runs the function 'number' times (default is often 1,000,000)
# and returns the total time taken as a float.
total_time = timeit.timeit(stmt="sine_wave_phasor(1, 1, 2, 10)", setup="from __main__ import sine_wave_phasor", number=1_000_000)
average_time = total_time / 1_000_000

print(f"Average time over 1,000,000 runs: {average_time:.6f} seconds")


Average time over 1,000,000 runs: 0.000006 seconds


why is it 3 times slower? I thought we wanted to make it faster!

Well, there is a second part to the trick! For sin(kx+wt+phi) we have an exponenitally increasing combinatorial as x and t increase in the simulation where sin() must be called.

For Im{ [A · e^{i(kx + φ)}] · e^{−iωt} }, we separate t and x and can precalculate them, only needing to recalculate when the frequency or starting positions change.

another trick we can do as programmers, is to precompute what we know we will need for our animation and store it prior to making the animation frames so that it is already ready when we need it, rather than having to calculate it each time on the fly.

*Question: what is memory that is allocated for precomputed, reusable values called?

* Answer cache

In [ ]:
import numpy as np
import timeit

def precompute_phasors(x, amps, phases_deg, disps, wavelength):
    k = 2*np.pi / wavelength
    phi = np.deg2rad(phases_deg)
    arg = k*(x[None, :] - disps[:, None]) + phi[:, None]
    # phasor for each wave at each x
    return amps[:, None] * np.exp(1j * arg)

def frames_traditional(x, amps, phases_deg, disps, wavelength, omega, dt, T):
    k = 2*np.pi / wavelength
    phi = np.deg2rad(phases_deg)
    out = 0.0
    for n in range(T):
        t = n * dt
        arg = k*(x[None, :] - disps[:, None]) + phi[:, None] - omega*t
        y = np.sum(amps[:, None] * np.sin(arg), axis=0)
        out += y[0]  # prevent optimization-away
    return out

def frames_phasor(C, omega, dt, T):
    # C is (M,N) complex, precomputed
    rot = np.exp(-1j * omega * dt)  # scalar
    r = 1.0 + 0.0j
    out = 0.0
    for _ in range(T):
        y = np.imag(np.sum(C * r, axis=0))  # sum waves, take imag
        out += y[0]
        r *= rot
    return out

# --- benchmark setup ---
N = 500 # number of spatial points (x)
M = 100 # number of superposed waves
T = 100 # number of frames/time steps
wavelength = 0.3
dt = 0.01
omega = 2*np.pi*3.0   # pick a frequency (3 Hz) for example

rng = np.random.default_rng(0)
x = np.linspace(0, 1, N)
amps = rng.random(M)
phases = rng.uniform(-180, 180, M)
disps = rng.random(M)

C = precompute_phasors(x, amps, phases, disps, wavelength)

# quick correctness spot-check for a couple frames
k = 2*np.pi / wavelength
phi = np.deg2rad(phases)
for n in [0, 5, 17]:
    t = n*dt
    yt = np.sum(amps[:, None] * np.sin(k*(x[None,:]-disps[:,None]) + phi[:,None] - omega*t), axis=0)
    yp = np.imag(np.sum(C * np.exp(-1j*omega*t), axis=0))
    assert np.allclose(yt, yp, atol=1e-12)

number = 10
t_trad = timeit.timeit(lambda: frames_traditional(x, amps, phases, disps, wavelength, omega, dt, T), number=number)
t_phas = timeit.timeit(lambda: frames_phasor(C, omega, dt, T), number=number)

print(f"traditional frames: {t_trad/number:.6e} s/run (T={T})")
print(f"phasor frames:      {t_phas/number:.6e} s/run (T={T})")
print(f"speedup (trad/phas): { (t_trad/t_phas):.3f}x")

traditional frames: 1.791611e-01 s/run (T=100)
phasor frames:      1.669295e-02 s/run (T=100)
speedup (trad/phas): 10.733x


In [ ]:
from os import setxattr
def precompute_r_grid(sources, X, Y, eps=1e-12):
  """
  Compute distances r_i(x,y) from each source to each grid point

  Args:
  - sources: (Ns, 2) array of [x,y] positions
  -X, Y: meshgrid arrays of shape (Ny, Nx)
  Note:
  - Ns means number of sources
  - Ny is number of y points in our grid
  - Nx is number of x points in our grid
Returns:
- r_grid: (Ns, Ny, Nx) array of distances
  """
  sx = sources[:,0] # get the x positions for all sources
  sy = sources[:,1] # get the y positions for all sources
  DX = X - sx # get the distance (difference) between source positions and grid positions
  DY = Y - sy # get the distance (diff) between source pos and grid pos
  r_grid = np.hypot(DX, DY) + eps # add a tiny eps to avoid division by zero

In [ ]:
def precompute_field_coefficient(r_grid, amplitudes, phases_rad, wavelength):
  """
  Precompute complex (imaginary) coefficients for each source and grid point.
  *remember the phasor trick:
  - A_i * sin(k*r - omega*t + phi_i) = Im( A_i * e^(i*(k*r + phi_i)) * e^(-i*omega*t) )
  - So  define C_i(x,y) = A_i * e^(i*(k*r + phi_i))
  - then field(x,y,t)  = Im ( sum_i C_i*e^(-i * omega*t) )
  --> per frame work becomes a single complex multiply and a sum instead of hundreds to thousands of sin() calls

    Inputs:
    - r_grid: (Ns, Ny, Nx)
    - amplidtudes: (Ns, )
    - phases_rad: (Ns, ) radians
    - wavelength: scalar
    Returns:
    --field_coeffs: complex array (Ns, Ny, Nx)
  """
  k = 2. *np.pi / float(wavelength)
  amp = amplitudes
  phi = phases_rad
  coeffs = amp * np.exp(1j* (k*r_grid+phi))
  return coeffs


In [ ]:
def evaluate_field(field_coeffs, omega, t):
  """
per-rame evalutation when filed_coeffs are precomputed:
- multply by e^(-i*omega*t)
- sum across all sources
- take just the imaginary part
inputs:
-fields_coeffs: (Ns, Ny, Nx)
- omega: amgular freq (scalar)
- t: time (scalar)
returns:
- filed (Ny, Nx) real-valued array
  """
  total = np.sum(np.exp(-1j*omega*t), axis=0)
  return np.imag(total)